In [2]:
import time
from threading import Event
from binance import ThreadedWebsocketManager
import pandas as pd
# need to import FuturesType
from binance.enums import FuturesType
import websocket
import json
import datetime
from threading import Thread

import os
import sys

sys.path.append("/home/info_core")
from etc.redis_connector.redis_connector import InitRedis

In [ ]:
redis = InitRedis(passwd="CommunityRedis123!", host='redis', port=6379, db=1)

In [3]:
def binance_websocket(store_dict, data, error_event, proc_name):
    def on_message(ws, message):
        if error_event.is_set():
            ws.close()
            raise Exception(f"binance_websocket|{proc_name} error_event is set. closing websocket..")
        msg = json.loads(message)
        print(msg)                                                    # test
        if 's' in msg.keys():
            store_dict[msg['s']] = {**msg, "last_update_timestamp": int(datetime.datetime.utcnow().timestamp()*1000000)}
    def on_error(ws, error):
        # print(f'binance_websocket on_error executed!')
        print(error)
        # self.websocket_logger.error(f'binance_websocket|binance_websocket on_error executed!\n Error: {error}')
        pass

    def on_close(ws, close_status_code, close_msg):
        print(f"\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")
        # self.websocket_logger.info(f"binance_websocket|\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")

    def on_open(ws):
        print(f'binance_websocket started')
        # self.websocket_logger.info(f'binance_websocket|binance_websocket started')
        ws.send(json.dumps(data))

    websocket.enableTrace(False)
    ws = websocket.WebSocketApp("wss://stream.binance.com/ws",
                            on_open=on_open,
                            on_message=on_message,
                            on_error=on_error,
                            on_close=on_close)
    ws.run_forever(ping_interval=30)

event = Event()
data = {
    "method": "SUBSCRIBE",
    "params":[
        "btcusdt@ticker",
    ],
    "id": 1
}

# binance_websocket({}, data, event)
binance_websocket_thread = Thread(target=binance_websocket, args=({}, data, event))
binance_websocket_thread.start()

binance_websocket started
{'result': None, 'id': 1}
{'e': '24hrTicker', 'E': 1700729913492, 's': 'BTCUSDT', 'p': '938.32000000', 'P': '2.567', 'w': '37089.25118596', 'x': '36551.10000000', 'c': '37489.43000000', 'Q': '0.01088000', 'b': '37489.42000000', 'B': '2.60070000', 'a': '37489.43000000', 'A': '3.70261000', 'o': '36551.11000000', 'h': '37861.10000000', 'l': '36239.20000000', 'v': '39222.71001000', 'q': '1454740943.75500250', 'O': 1700643513491, 'C': 1700729913491, 'F': 3291002661, 'L': 3292494393, 'n': 1491733}
{'e': '24hrTicker', 'E': 1700729914523, 's': 'BTCUSDT', 'p': '938.32000000', 'P': '2.567', 'w': '37089.25173259', 'x': '36551.10000000', 'c': '37489.42000000', 'Q': '0.00811000', 'b': '37489.42000000', 'B': '2.59259000', 'a': '37489.43000000', 'A': '3.99086000', 'o': '36551.10000000', 'h': '37861.10000000', 'l': '36239.20000000', 'v': '39222.68431000', 'q': '1454740012.00149790', 'O': 1700643514523, 'C': 1700729914523, 'F': 3291002664, 'L': 3292494394, 'n': 1491731}
{'e': 

In [ ]:
redis.close_all()

In [ ]:
usdm_symbol_list = ['BTCUSDT']
coinm_symbol_list = ['BTCUSD_PERP']
spot_symbol_list = ['BTCUSDT']
twm = ThreadedWebsocketManager()
# start is required to initialise its internal loop
twm.start()

def handle_spot_kline_msg(msg):
    if msg['data']['k']['x'] == True:
        print(msg['data'])

# twm.start_kline_socket(callback=handle_socket_message, symbol=symbol)

# multiple sockets can be started
# twm.start_depth_socket(callback=handle_socket_message, symbol=symbol)

# or a multiplex socket can be started like this
# see Binance docs for stream names
# usdm_bookticker_streams = [x.lower()+'@bookTicker' for x in usdm_symbol_list]
# usdm_ticker_streams = [x.lower()+'@ticker' for x in usdm_symbol_list]
# coinm_bookticker_streams = [x.lower()+'@bookTicker' for x in coinm_symbol_list]
# spot_bookticker_streams = [x.lower()+'@bookTicker' for x in spot_symbol_list]
# twm.start_futures_multiplex_socket(callback=handle_usdm_liquidation_msg, streams=['!forceOrder@arr'])
# twm.start_futures_multiplex_socket(callback=handle_coinm_liquidation_msg, streams=['!forceOrder@arr'], futures_type=FuturesType.COIN_M)
# twm.start_futures_multiplex_socket(callback=handle_coinm_msg, streams=coinm_streams, futures_type=FuturesType.COIN_M)
twm.start_multiplex_socket(callback=handle_spot_kline_msg, streams=[x.lower()+'@kline_1m' for x in spot_symbol_list]+[x.lower()+'@kline_3m' for x in spot_symbol_list]+[x.lower()+'@kline_5m' for x in spot_symbol_list])



twm.join()

In [ ]:
twm.stop()

In [ ]:
from binance_websocket import BinanceWebsocket
import pandas as pd

In [ ]:
binance_spot_symbol_list = ['BTCUSDT','ETHUSDT','XRPUSDT','BCHUSDT','XEMUSDT','MTLUSDT','KAVAUSDT','TUSDT','SUIUSDT','SOLUSDT','WAVESUSDT','STXUSDT','STORJUSDT','DOGEUSDT','ETCUSDT','FLOWUSDT','ETCUSDT','MATICUSDT','AAVEUSDT','ALGOUSDT','GMTUSDT','SANDUSDT','ANKRUSDT','ONTUSDT','SXPUSDT']
binance_usdm_symbol_list = binance_spot_symbol_list
binance_coinm_symbol_list = [x[:-1]+'_PERP' for x in binance_spot_symbol_list]
proc_n = 3
logging_dir = '/home/kp_info_loader/exchange_websocket/'
binance_ws = BinanceWebsocket(binance_spot_symbol_list, binance_usdm_symbol_list, binance_coinm_symbol_list, proc_n, logging_dir)

In [ ]:
binance_ws.binance_spot_kline_queue_dict['BTCUSDT'].get()

In [ ]:
binance_ws.binance_websocket_proc_dict

In [ ]:
binance_ws.check_status(print_result=True)

In [ ]:
list(binance_ws.binance_usdm_liquidation_list)

In [ ]:
list(binance_ws.binance_coinm_liquidation_list)

In [ ]:
dict(binance_ws.binance_usdm_bookticker_dict)

In [ ]:
test_dict = {'a': 1, 'b': 2}

In [ ]:
len(test_dict)

In [ ]:
binance_ws.check_status(print_result=True)

In [ ]:
binance_ws.terminate_websocket()